# Task 1 — NISAR Dataset Exploration for Brahmaputra FloodWatch
**NASA Space Apps Challenge 2026** | **Challenge:** *Dancing with the SARs*

This notebook demonstrates how to inspect, unpack, and visualize official **NASA-ISRO NISAR Level-2 GCOV (Geocoded Polarimetric Covariance)** HDF5 radar products.

In [ ]:
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.transform import from_bounds

DATA_PATH = os.path.join("..", "data", "nisar_products", "NISAR_S2_PR_GCOV_025_019_D_075_2500_CRNA_A_20260709T124208_20260709T124245_P00500_M_F_I_001.h5")
print("Target NISAR file exists:", os.path.exists(DATA_PATH))

## 1. Inspect HDF5 Group Hierarchy & CF-1.8 Attributes

In [ ]:
with h5py.File(DATA_PATH, 'r') as h5:
    print("=== CF Root Metadata ===")
    for k, v in h5.attrs.items():
        print(f"{k}: {v}")
        
    print("\n=== Discovered Datasets ===")
    def list_items(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f"Dataset: {name:<45} | shape: {str(obj.shape):<15} | dtype: {obj.dtype}")
    h5.visititems(list_items)

## 2. Read Calibrated Backscatter & Coordinates

In [ ]:
with h5py.File(DATA_PATH, 'r') as h5:
    gcov = h5['science/SSAR/GCOV']
    freq_a = gcov['grids/frequencyA']
    
    hh_linear = freq_a['HHHH'][:]
    hv_linear = freq_a['HVHV'][:]
    mask = freq_a['mask'][:]
    x_coords = freq_a['xCoordinates'][:]
    y_coords = freq_a['yCoordinates'][:]
    proj = freq_a['projection']
    epsg = proj.attrs['epsg_code']

# Handle validity mask (0 = valid pixel)
valid_hh = np.where(mask == 0, hh_linear, np.nan)
valid_hh = np.where(valid_hh > 1e-7, valid_hh, 1e-7)

# Convert to decibels (dB)
hh_db = 10.0 * np.log10(valid_hh)

print(f"CRS: EPSG:{epsg}")
print(f"HH backscatter range: {np.nanmin(hh_db):.2f} dB to {np.nanmax(hh_db):.2f} dB")
print(f"Mean backscatter: {np.nanmean(hh_db):.2f} dB")

## 3. Scientific Grayscale Visualization

In [ ]:
plt.figure(figsize=(9, 8), dpi=120)
p1, p99 = np.nanpercentile(hh_db, 1), np.nanpercentile(hh_db, 99)
plt.imshow(hh_db, cmap='gray', vmin=p1, vmax=p99)
cbar = plt.colorbar(fraction=0.046, pad=0.04)
cbar.set_label('Gamma-0 Backscatter (dB)')
plt.title('NISAR S-band L2 GCOV (Co-Pol HH) — 09 July 2026\nBrahmaputra Flood Corridor, Assam [EPSG:32646]')
plt.xlabel('Pixel Column (25m posting)')
plt.ylabel('Pixel Row (25m posting)')
plt.show()